# Visualize CLIP Embeddings

This notebook projects image embeddings to 2D using PCA or UMAP and renders an interactive Plotly scatter plot.

In [1]:
import numpy as np
import pandas as pd
import plotly.express as px
from sklearn.decomposition import PCA

In [2]:
# Paths
from pathlib import Path

BASE_DIR = Path.cwd()
DATA_DIR = BASE_DIR.parent / "data"
EMBED_DIR = DATA_DIR / "image_embeddings"

EMBEDDINGS_PATH = EMBED_DIR / "image_embeddings.npy"
IDS_PATH = EMBED_DIR / "product_ids.npy"
PARQUET_PATH = DATA_DIR / "women_data.parquet"

# Choose label: "product_category" or "color"
LABEL_COL = "product_category"

# Method: "pca" or "umap"
METHOD = "pca"

In [3]:
embeddings = np.load(EMBEDDINGS_PATH)
product_ids = np.load(IDS_PATH, allow_pickle=True).astype(str)

df = pd.read_parquet(PARQUET_PATH)
if "row_id" not in df.columns:
    raise ValueError("row_id column not found in parquet")
df["row_id"] = df["row_id"].astype(str)
df = df.set_index("row_id")

labels = []
keep_idx = []
for i, pid in enumerate(product_ids):
    if pid in df.index:
        labels.append(str(df.loc[pid, LABEL_COL]))
        keep_idx.append(i)

if not keep_idx:
    raise ValueError("No matching ids found between embeddings and parquet")

X = embeddings[keep_idx]
labels = np.array(labels)
len(X), X.shape

(1116, (1116, 512))

In [4]:
if METHOD == "umap":
    import umap
    reducer = umap.UMAP(n_components=2, random_state=42)
    X2 = reducer.fit_transform(X)
else:
    X2 = PCA(n_components=2, random_state=42).fit_transform(X)

X2.shape

(1116, 2)

In [6]:
plot_df = pd.DataFrame(
    {
        "x": X2[:, 0],
        "y": X2[:, 1],
        "label": labels,
        "row_id": product_ids[keep_idx],
    }
)

fig = px.scatter(
    plot_df,
    x="x",
    y="y",
    color="label",
    hover_data=["row_id", "label"],
    title=f"Embeddings ({METHOD.upper()}) colored by {LABEL_COL}",
)
fig.show()